ZINGMP3

In [1]:
import requests
import pandas as pd
import re
from collections import defaultdict  

def extract_album_id(album_link):
    match = re.search(r'/album/.*?/([\w-]+)\.html', album_link or "")
    return match.group(1) if match else "N/A"

def determine_album_type(track_count):
    if track_count <= 3:
        return "Single"
    elif 4 <= track_count <= 6:
        return "EP"
    else:
        return "Regular"

def fetch_artist_songs(artist_name):
    url = f"http://localhost:5000/api/artistsongs?name={artist_name}"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        songs = data.get("songs", [])

        records = []
        album_tracks = defaultdict(set)  # Lưu các track duy nhất trong mỗi album

        for song in songs:
            album_id = extract_album_id(song.get("albumLink", "") or "")
            album_name = song.get("album", song.get("title", "N/A"))
            tracklist = song.get("tracklist", [])

            if not tracklist:
                tracklist = [{"title": song.get("title", "N/A"), "link": song.get("link", "N/A")}]

            for track in tracklist:
                track_title = track.get("title", "N/A")
                album_tracks[album_id].add(track_title)  # Lưu các track vào set để tránh trùng lặp

                records.append({
                    "album_id": album_id,
                    "album_name": album_name,
                    "tracklist": track_title,
                    "release_date": song.get("releaseDate", "Unknown"),
                    "provided_by": song.get("providedBy", "Unknown"),
                    "featured_artists": song.get("featuredArtists", "Unknown"),
                    "album_artist": song.get("albumOwner", "Unknown"),
                    "ZingMP3": track.get("link", "N/A")
                })

        df = pd.DataFrame(records)
        df = df.drop_duplicates(subset=["album_id", "tracklist"])  # Tránh bài hát trùng lặp
        df = df.sort_values(by=["album_name", "tracklist"], ascending=[True, True])

        # Thêm cột phân loại album dựa trên số lượng track thực tế
        df["album_type"] = df["album_id"].map(lambda x: determine_album_type(len(album_tracks[x])))

        return df
    else:
        print("Error fetching data:", response.status_code)
        return None
artist_name = 'Binh-Gold'
df = fetch_artist_songs(artist_name)
df.to_excel(f'{artist_name}_songZingMP3.xlsx', index=False)
df.head()

,album_id,album_name,tracklist,release_date,provided_by,featured_artists,album_artist,ZingMP3,album_type
7,6C9IED88,ADAMN (Single),ADAMN,10/04/2025,MIXUS,Bình Gold,Bình Gold,https://zingmp3.vn/bai-hat/ADAMN-Binh-Gold/Z8U...,Single
9,N/A,"Em Iu (Dustee, Monotape & Teddy Doox Remix)","Em Iu (Dustee, Monotape & Teddy Doox Remix)",None,None,"Wxrdie, Andree Right Hand, Bình Gold",None,https://zingmp3.vn/bai-hat/Em-Iu-Dustee-Monota...,Single
5,6BZWUBO6,Em iu (Max Benderz Remix) (Single),Em iu (Max Benderz Remix),30/01/2022,Believe,"Andree Right Hand, Wxrdie, Bình Gold, 2Pillz","Andree Right Hand, Wxrdie, Bình Gold, 2Pillz",https://zingmp3.vn/bai-hat/Em-iu-Max-Benderz-R...,Single
1,6B6W6EDA,Em iu (Single),Em iu,30/01/2022,Believe,"Andree Right Hand, Wxrdie, Bình Gold, 2Pillz","Andree Right Hand, Wxrdie, Bình Gold, 2Pillz",https://zingmp3.vn/bai-hat/Em-iu-Andree-Right-...,Single
10,6B7FWBIF,Em iu (Single),Em iu,09/01/2023,Believe,"Wind.P, Andree Right Hand, Wxrdie, Bình Gold","Wind.P, Andree Right Hand, Wxrdie, Bình Gold",https://zingmp3.vn/bai-hat/Em-iu-Wind-P-Andree...,Single


SPOTIFY



In [ ]:
import spotipy
import pandas as pd
from spotipy.oauth2 import SpotifyClientCredentials

# Hàm xác định loại album
def determine_album_type(track_count):
    if track_count <= 3:
        return "Single"
    elif 4 <= track_count <= 6:
        return "EP"
    else:
        return "Regular"

# Hàm lấy tất cả bài hát của nghệ sĩ
def get_artist_tracks_all(artist_name):
    client_id = "c7e1fe3ffe674920a01f9b016e6ae5df"
    client_secret = "215c6808dea74656b3629b306182ac4b"
    sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=client_id, client_secret=client_secret))

    # 🔎 Tìm nghệ sĩ theo tên
    results = sp.search(q=artist_name, type='artist', limit=1)
    if not results['artists']['items']:
        print("Không tìm thấy nghệ sĩ.")
        return pd.DataFrame()

    artist_id = results['artists']['items'][0]['id']

    track_data = []
    seen_tracks = set()
    album_tracks_count = {}


    offset = 0
    while True:
        albums = sp.artist_albums(artist_id, album_type='album,single', limit=50, offset=offset)
        if not albums['items']:
            break
        for album in albums['items']:
            album_id = album['id']
            album_name = album['name']
            album_release_date = album.get('release_date', 'Unknown')
            album_owner = album['artists'][0]['name']

            album_info = sp.album(album_id)
            label = album_info.get('label', 'Unknown')

           
            tracks = sp.album_tracks(album_id)['items']
            album_tracks_count[album_name] = len(tracks)  # Ghi lại số lượng bài hát

            for track in tracks:
                track_id = track['id']
                if track_id in seen_tracks:
                    continue
                seen_tracks.add(track_id)

                track_title = track['name']
                link_spotify = track['external_urls']['spotify']
                featured_artists = [artist['name'] for artist in track['artists'] if artist['id'] != artist_id]
                featured_artists = ", ".join(featured_artists) if featured_artists else "None"

                # Xử lý ngày phát hành
                try:
                    album_release_date = pd.to_datetime(album_release_date, errors='coerce').strftime("%d/%m/%Y")
                except:
                    album_release_date = "Unknown"

                track_data.append([album_name, track_title, album_release_date, featured_artists, album_owner, label, link_spotify])
        offset += 50  

   
    top_tracks = sp.artist_top_tracks(artist_id, country="US")['tracks']
    for track in top_tracks:
        track_id = track['id']
        if track_id in seen_tracks:
            continue
        seen_tracks.add(track_id)

        track_title = track['name']
        link_spotify = track['external_urls']['spotify']
        album = track['album']
        album_name = album['name']
        album_release_date = album.get('release_date', 'Unknown')
        album_owner = album['artists'][0]['name']
        featured_artists = [artist['name'] for artist in track['artists'] if artist['id'] != artist_id]
        featured_artists = ", ".join(featured_artists) if featured_artists else "None"

        # 🔹 Lấy thông tin label của album
        album_info = sp.album(album['id'])
        label = album_info.get('label', 'Unknown')

        try:
            album_release_date = pd.to_datetime(album_release_date, errors='coerce').strftime("%d/%m/%Y")
        except:
            album_release_date = "Unknown"

        track_data.append([album_name, track_title, album_release_date, featured_artists, album_owner, label, link_spotify])
        album_tracks_count[album_name] = album_tracks_count.get(album_name, 0) + 1  # Cập nhật số bài hát


    columns = ["album_name", "tracklist", "release_date", "featured_artists", "album_owner_", "provided_by", "Link_Spotify"]
    df = pd.DataFrame(track_data, columns=columns)

    df['album_type'] = df['album_name'].map(lambda x: determine_album_type(album_tracks_count.get(x, 0)))
    df = df.sort_values(by=["album_name", "tracklist"], ascending=[True, True])
 

    return df

# Chạy chương trình
if __name__ == "__main__":
    artist_name_Spotify = 'Binh-Gold'
    df_tracks = get_artist_tracks_all(artist_name_Spotify)



In [3]:
df_tracks.head()

,album_name,tracklist,release_date,featured_artists,album_owner_,provided_by,Link_Spotify,album_type
0,ADAMN,ADAMN,10/04/2025,None,Bình Gold,Binh Gold,https://open.spotify.com/track/50WjeoNacrSiDum...,Single
15,BCDBL,BCDBL,02/02/2022,None,Bình Gold,Binh Gold,https://open.spotify.com/track/1Vi2Hln3U0kwCob...,Single
14,Bật Chế Độ Bay Lên (Huy Lee Remix),Bật Chế Độ Bay Lên (Short Version) - Huy Lee R...,04/07/2022,None,Bình Gold,Bình Gold,https://open.spotify.com/track/7dPvKrYoFMd8QTs...,Single
13,Bật Chế Độ Bay Lên (Huy Lee Remix),Bật Chế Độ Bay Lên - Huy Lee Remix,07/04/2022,None,Bình Gold,Bình Gold,https://open.spotify.com/track/3eZcAYvAIVnRIJ8...,Single
17,Bịt Khẩu Trang Vào,Bịt Khẩu Trang Vào,07/02/2021,None,Bình Gold,Binh Gold,https://open.spotify.com/track/2gM6Ld7bA28ULuN...,Single


ZINGMP3+SPOTIFY

In [4]:
import pandas as pd
from thefuzz import process

# Chọn các cột cần thiết từ cả hai nguồn
df_tracks = df_tracks[
    [
        'album_name', 'tracklist', 'release_date', 'featured_artists',
        'album_owner_', 'provided_by', 'Link_Spotify', 'album_type'
    ]
]

df = df[
    [
        'album_id', 'album_name', 'tracklist', 'release_date', 'provided_by',
        'featured_artists', 'album_artist', 'ZingMP3', 'album_type'
    ]
]

# Tạo danh sách các tên album từ nguồn ZingMP3
album_names_zing = df['album_name'].tolist()

def find_best_match(album_name):
    result = process.extractOne(album_name, album_names_zing, score_cutoff=85)
    if result:  
        match, score = result  
        return match
    return album_name  

# Áp dụng fuzzy matching vào dữ liệu Spotify
df_tracks['album_name_matched'] = df_tracks['album_name'].apply(find_best_match)

# Gộp dữ liệu dựa trên album_name_matched, tracklist, album_type
df_merged = pd.merge(
    df_tracks, df, left_on=["album_name_matched", "tracklist", "album_type"], 
    right_on=["album_name", "tracklist", "album_type"], how="outer", suffixes=("_Spotify", "_ZingMP3")
)

# **Chọn tên album ưu tiên theo Spotify, nếu không có thì lấy từ ZingMP3**
df_merged["album_name_final"] = df_merged["album_name_Spotify"].combine_first(df_merged["album_name_ZingMP3"])

# Đổi tên cột
df_ = df_merged.rename(columns={
    "album_name_final": "album_name",
    "tracklist": "tracklist(danh sách bài hát)",
    "featured_artists_Spotify": "Song artist(nghệ sĩ tham gia bài hát)(Spotify)",
    "album_owner_Spotify": "Album artist (nghệ sĩ sở hữu album)*(Spotify)",
    "release_date_Spotify": "Ngày phát hành trên Spotify",
    "provided_by_ZingMP3": "Cung cấp bởi(ZingMP3)",
    "provided_by_Spotify": "Cung cấp bởi(Spotify)",
    "featured_artists_ZingMP3": "Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)",
    "album_owner_ZingMP3": "Album artist (nghệ sĩ sở hữu album)*(ZingMP3)",
    "release_date_ZingMP3": "Ngày phát hành trên ZingMP3",
    "ZingMP3": "Link_ZingMP3",
    "Link_Spotify": "Spotify",
    "album_id": "Mã định danh album ZingMP3"
})

# Chọn các cột cần thiết
desired_columns = [
    "album_name",
    "album_owner_",  
    "album_type",
    "tracklist(danh sách bài hát)",
    "Ngày phát hành trên Spotify",
    "Ngày phát hành trên ZingMP3",
    "Song artist(nghệ sĩ tham gia bài hát)(Spotify)",
    "Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)",
    "Cung cấp bởi(ZingMP3)", 
    "Cung cấp bởi(Spotify)", 
    "Mã định danh album ZingMP3",
    "Link_ZingMP3",
    "Spotify"
]

df_ = df_[desired_columns]

# Xuất file Excel
df_.to_excel(f"{artist_name}_song_ZingMP3+Spotify.xlsx", index=False)
df_.head()


,album_name,album_owner_,album_type,tracklist(danh sách bài hát),Ngày phát hành trên Spotify,Ngày phát hành trên ZingMP3,Song artist(nghệ sĩ tham gia bài hát)(Spotify),Song artist(nghệ sĩ tham gia bài hát)(ZingMP3),Cung cấp bởi(ZingMP3),Cung cấp bởi(Spotify),Mã định danh album ZingMP3,Link_ZingMP3,Spotify
0,ADAMN,Bình Gold,Single,ADAMN,10/04/2025,10/04/2025,None,Bình Gold,MIXUS,Binh Gold,6C9IED88,https://zingmp3.vn/bai-hat/ADAMN-Binh-Gold/Z8U...,https://open.spotify.com/track/50WjeoNacrSiDum...
1,BCDBL,Bình Gold,Single,BCDBL,02/02/2022,NaN,None,NaN,NaN,Binh Gold,NaN,NaN,https://open.spotify.com/track/1Vi2Hln3U0kwCob...
2,Bịt Khẩu Trang Vào,Bình Gold,Single,Bịt Khẩu Trang Vào,07/02/2021,NaN,None,NaN,NaN,Binh Gold,NaN,NaN,https://open.spotify.com/track/2gM6Ld7bA28ULuN...
3,Bốc Bát Họ,Bình Gold,Single,Bốc Bát Họ,04/08/2018,NaN,None,NaN,NaN,Binh Gold,NaN,NaN,https://open.spotify.com/track/42vo9vsUghqqqnI...
4,DBAH,Bình Gold,Single,DBAH,16/09/2022,NaN,OgeNus,NaN,NaN,Binh Gold,NaN,NaN,https://open.spotify.com/track/3ZHqrVIOWsGlKa0...


APPLE MUSIC

In [5]:
import jwt  # Install with: pip install pyjwt
import time
import requests
import pandas as pd
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

TEAM_ID = '6A3CSCZ9M6'
KEY_ID = '2VLJH6R856'
PRIVATE_KEY_PATH = r'D:\AuthKey_2VLJH6R856.p8'

def generate_apple_music_token():
    with open(PRIVATE_KEY_PATH, "r") as key_file:
        private_key = key_file.read()
    
    payload = {
        "iss": TEAM_ID,
        "iat": int(time.time()),
        "exp": int(time.time()) + 3600,  
    }

    token = jwt.encode(payload, private_key, algorithm="ES256", headers={"alg": "ES256", "kid": KEY_ID})
    return token

APPLE_MUSIC_TOKEN = generate_apple_music_token()

In [7]:
def get_artist_albums(artist_id, storefront="us"):
    """ Lấy danh sách album của nghệ sĩ """
    url = f"https://api.music.apple.com/v1/catalog/{storefront}/artists/{artist_id}/albums"
    headers = {"Authorization": f"Bearer {APPLE_MUSIC_TOKEN}"}

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        albums = response.json().get("data", [])
        return [
            {
                "album_id": album["id"],
                "album_name": album["attributes"]["name"],
                "release_date": album["attributes"]["releaseDate"],
                "medium": "single" if album["attributes"]["isSingle"] else "album",
                "genre": ", ".join(album["attributes"].get("genreNames", [])),
                "album_url": album["attributes"]["url"],
                "label": album["attributes"].get("recordLabel", "Unknown")
            }
            for album in albums
        ]
    return []

def get_album_tracks(album_id, storefront="us"):
    """ Lấy danh sách bài hát trong album """
    url = f"https://api.music.apple.com/v1/catalog/{storefront}/albums/{album_id}/tracks"
    headers = {"Authorization": f"Bearer {APPLE_MUSIC_TOKEN}"}

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        tracks = response.json().get("data", [])
        return [
            {
                "album_id": album_id,
                "tracklist(danh sách bài hát)": track["attributes"]["name"],
                "track_url": track["attributes"]["url"],
                "featured_artists": ", ".join(
                    [artist["attributes"]["name"] for artist in track.get("relationships", {}).get("artists", {}).get("data", [])]
                ) if track.get("relationships", {}).get("artists") else track["attributes"].get("artistName", "None"),
            }
            for track in tracks
        ]
    return []

artist_ids = ["1486157747"]

# 🔹 Lấy thông tin album của nghệ sĩ
albums_data = []
for artist_id in artist_ids:
    albums_data.extend(get_artist_albums(artist_id))

df_albums = pd.DataFrame(albums_data)

# 🔹 Lấy thông tin bài hát của album đồng thời
tracks_data = []
with ThreadPoolExecutor() as executor:
    results = executor.map(lambda album: get_album_tracks(album["album_id"]), albums_data)

for album, track_list in zip(albums_data, results):
    for track in track_list:
        track.update({
            "album_name": album["album_name"],
            "release_date": album["release_date"],
            "status_code": "Normal",
            "class": "digital",
            "genre": album["genre"],
            "medium": album["medium"],
            "label": album["label"]

        })
        tracks_data.append(track)

df_tracks_apple = pd.DataFrame(tracks_data)
df_tracks_apple.head()

,album_id,tracklist(danh sách bài hát),track_url,featured_artists,album_name,release_date,status_code,class,genre,medium,label
0,1606483780,"Em iu (feat. Wxrdie, Bình Gold & 2pillz)",https://music.apple.com/us/album/em-iu-feat-wx...,Andree Right Hand,"Em iu (feat. Wxrdie, Bình Gold & 2pillz) - Single",2022-01-30,Normal,digital,"Hip-Hop/Rap, Music",single,$maker Music
1,1771437409,Đổi Tư Thế,https://music.apple.com/us/album/%C4%91%E1%BB%...,Bình Gold & Andree Right Hand,Đổi Tư Thế - Single,2024-10-02,Normal,digital,"Hip-Hop/Rap, Music",single,Binh Gold
2,1748724965,OBGTLH,https://music.apple.com/us/album/obgtlh/174872...,Bình Gold & Lil Shady,OBGTLH - Single,2020-08-06,Normal,digital,"Hip-Hop/Rap, Music",single,Binh Gold
3,1748785808,Trăng Hoa Mây Mưa,https://music.apple.com/us/album/tr%C4%83ng-ho...,Bình Gold,Trăng Hoa Mây Mưa - Single,2024-01-13,Normal,digital,"Hip-Hop/Rap, Music",single,Binh Gold
4,1749301687,Tuổi Gì Mà Chẳng Thích Lì Xì (feat. Bình Gold),https://music.apple.com/us/album/tu%E1%BB%95i-...,Bích Phương,Tuổi Gì Mà Chẳng Thích Lì Xì (feat. Bình Gold)...,2020-01-16,Normal,digital,"Pop, Music",single,MUNIZ MUSIC COMPANY LIMITED


ZINGMP3+SPOTIFY+APPLE


In [8]:
import pandas as pd
from fuzzywuzzy import fuzz, process

# Chuẩn hóa văn bản (Title Case)
def normalize_text(text):
    if isinstance(text, str):
        return text.strip().lower().title()
    return text

# Hàm fuzzy match để tìm match gần nhất
def fuzzy_match_single(value, choices):
    best_match = process.extractOne(value, choices, scorer=fuzz.token_sort_ratio)
    return best_match[0] if best_match else None

# Chuẩn hóa văn bản cho các cột tên album và bài hát
df_["album_name"] = df_["album_name"].apply(normalize_text)
df_tracks_apple["album_name"] = df_tracks_apple["album_name"].apply(normalize_text)
df_["tracklist(danh sách bài hát)"] = df_["tracklist(danh sách bài hát)"].apply(normalize_text)
df_tracks_apple["tracklist(danh sách bài hát)"] = df_tracks_apple["tracklist(danh sách bài hát)"].apply(normalize_text)

# Đảm bảo cột tracklist được đặt tên đúng
df_tracks_apple = df_tracks_apple.rename(columns={"tracklist": "tracklist(danh sách bài hát)"})

# Fuzzy match cho album_name giữa Apple Music và Zing/Spotify
df_tracks_apple['album_name_fuzzy'] = df_tracks_apple['album_name'].apply(
    lambda x: fuzzy_match_single(x, df_['album_name'])
)

# Merge theo tên album đã fuzzy match và tên bài hát
df_final = pd.merge(
    df_, 
    df_tracks_apple, 
    left_on=["album_name", "tracklist(danh sách bài hát)"], 
    right_on=["album_name_fuzzy", "tracklist(danh sách bài hát)"], 
    how="outer", 
    suffixes=("", "_Apple")
)

# Thêm cột "Cung cấp bởi(AppleMusic)" từ label nếu có
if "label" in df_tracks_apple.columns:
    df_final["Cung cấp bởi(AppleMusic)"] = df_final["label"]
else:
    df_final["Cung cấp bởi(AppleMusic)"] = ""
# Điền album_name còn thiếu từ AppleMusic (album_name_fuzzy)
df_final['album_name'] = df_final['album_name'].combine_first(df_final['album_name_fuzzy'])

# Xóa các cột không cần thiết
columns_to_drop = ["album_name_fuzzy", "album_id", "status_code", "class", "medium"]

df_final.drop(columns=[col for col in columns_to_drop if col in df_final.columns], inplace=True)


# Xóa trùng lặp
df_final.drop_duplicates(subset=["album_name", "tracklist(danh sách bài hát)"], keep="first", inplace=True)

# Đổi tên các cột theo yêu cầu
df_final = df_final.rename(columns={
    "album_owner_": "Nghệ sĩ sở hữu album", 
    "album_type": "album_type", 
    "genre": "genre", 
    "release_date": "Ngày phát hành trên AppleMusic", 
    "Song artist(nghệ sĩ tham gia bài hát)(Spotify)": "Song artist(nghệ sĩ tham gia bài hát)(Spotify)", 
    "Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)": "Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)", 
    "featured_artists": "Song artist(nghệ sĩ tham gia bài hát)(AppleMusic)", 
    "Cung cấp bởi(ZingMP3)": "Cung cấp bởi(ZingMP3)", 
    "Cung cấp bởi(Spotify)": "Cung cấp bởi(Spotify)", 
    "Link_ZingMP3": "ZingMP3", 
    "Spotify": "Spotify", 
    "track_url": "Apple Music"
})

# Chuyển định dạng ngày AppleMusic
df_final["Ngày phát hành trên AppleMusic"] = pd.to_datetime(
    df_final["Ngày phát hành trên AppleMusic"], errors='coerce'
).dt.strftime('%d/%m/%Y')

# Các cột cần xuất ra
final_columns = [
    'album_name', 'Nghệ sĩ sở hữu album', 'album_type', 'genre',
    'tracklist(danh sách bài hát)', 'Ngày phát hành trên Spotify',
    'Ngày phát hành trên ZingMP3', 'Ngày phát hành trên AppleMusic',
    'Song artist(nghệ sĩ tham gia bài hát)(Spotify)',
    'Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)',
    'Song artist(nghệ sĩ tham gia bài hát)(AppleMusic)',
    'Cung cấp bởi(ZingMP3)', 'Cung cấp bởi(Spotify)', 'Cung cấp bởi(AppleMusic)',
    'Mã định danh ZingMP3',  
    'ZingMP3', 'Spotify', 'Apple Music'
]

# Lọc giữ lại các cột
df_final = df_final[[col for col in final_columns if col in df_final.columns]]

# Xuất file Excel
df_final.to_excel(f"{artist_name}_AlbumsZingMp3_Spot_Apple.xlsx", index=False)

# Xem trước dữ liệu
df_final.head()


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


,album_name,Nghệ sĩ sở hữu album,album_type,genre,tracklist(danh sách bài hát),Ngày phát hành trên Spotify,Ngày phát hành trên ZingMP3,Ngày phát hành trên AppleMusic,Song artist(nghệ sĩ tham gia bài hát)(Spotify),Song artist(nghệ sĩ tham gia bài hát)(ZingMP3),Song artist(nghệ sĩ tham gia bài hát)(AppleMusic),Cung cấp bởi(ZingMP3),Cung cấp bởi(Spotify),Cung cấp bởi(AppleMusic),ZingMP3,Spotify,Apple Music
0,Adamn,Bình Gold,Single,NaN,Adamn,10/04/2025,10/04/2025,NaN,None,Bình Gold,NaN,MIXUS,Binh Gold,NaN,https://zingmp3.vn/bai-hat/ADAMN-Binh-Gold/Z8U...,https://open.spotify.com/track/50WjeoNacrSiDum...,NaN
1,Bcdbl,Bình Gold,Single,NaN,Bcdbl,02/02/2022,NaN,NaN,None,NaN,NaN,NaN,Binh Gold,NaN,NaN,https://open.spotify.com/track/1Vi2Hln3U0kwCob...,NaN
2,Bật Chế Độ Bay Lên (Huy Lee Remix),Bình Gold,Single,NaN,Bật Chế Độ Bay Lên (Short Version) - Huy Lee R...,04/07/2022,NaN,NaN,None,NaN,NaN,NaN,Bình Gold,NaN,NaN,https://open.spotify.com/track/7dPvKrYoFMd8QTs...,NaN
3,Bật Chế Độ Bay Lên (Huy Lee Remix),Bình Gold,Single,NaN,Bật Chế Độ Bay Lên - Huy Lee Remix,07/04/2022,NaN,NaN,None,NaN,NaN,NaN,Bình Gold,NaN,NaN,https://open.spotify.com/track/3eZcAYvAIVnRIJ8...,NaN
4,Bịt Khẩu Trang Vào,Bình Gold,Single,"Hip-Hop/Rap, Music",Bịt Khẩu Trang Vào,07/02/2021,NaN,07/02/2021,None,NaN,Bình Gold,NaN,Binh Gold,Binh Gold,NaN,https://open.spotify.com/track/2gM6Ld7bA28ULuN...,https://music.apple.com/us/album/b%E1%BB%8Bt-k...


YT MUSIC


In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# Khởi tạo WebDriver
driver = webdriver.Chrome()

# Mở trang YouTube Music của kênh
main_url = 'https://music.youtube.com/channel/UCUbaGyvUEx-lpEWpIX3_HWg'
driver.get(main_url)

# Đảm bảo trang đã tải đầy đủ
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
time.sleep(3)

# Danh sách lưu dữ liệu bài hát
data = []
album_map = {}  # Lưu ánh xạ list_id -> album_name

# --- PHẦN 1: Lấy bài hát từ trang chính của kênh ---
try:
    show_all_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, '//button[.//span[contains(text(), "Show all")]]'))
    )
    ActionChains(driver).move_to_element(show_all_button).click().perform()
    print("Đã click 'Show all'")
    time.sleep(3)
except Exception as e:
    print("Không tìm thấy hoặc không thể click nút 'Show all':", e)

# Tìm tất cả liên kết bài hát ở trang chính
song_links = driver.find_elements(By.XPATH, '//a[@class="yt-simple-endpoint style-scope yt-formatted-string"]')
for song in song_links:
    song_url = song.get_attribute('href')
    song_title = song.text
    if song_url and 'watch?v=' in song_url:
        # Trích xuất list ID nếu có
        list_id = None
        if 'list=' in song_url:
            list_id = song_url.split('list=')[1].split('&')[0]
        album_name = song_title  # Mặc định nếu không tìm thấy album
        if list_id and list_id in album_map:
            album_name = album_map[list_id]
        data.append({
            "tracklist": song_title,  # Đổi từ 'title' thành 'tracklist'
            "video_url": song_url,
            "album": album_name
        })

# Quay lại trang chính để xử lý phần 2
driver.get(main_url)
time.sleep(3)

# --- PHẦN 2: Lấy bài hát từ các album (SINGLE & EPs) ---
try:
    more_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='More']"))
    )
    more_button.click()
    time.sleep(3)
except Exception as e:
    print("Không tìm thấy nút 'More' cho SINGLE & EPs:", e)

# Tìm các liên kết album và tên album
album_elements = driver.find_elements(By.XPATH, "//a[@class='yt-simple-endpoint image-wrapper style-scope ytmusic-two-row-item-renderer']")
album_titles = driver.find_elements(By.XPATH, "//yt-formatted-string[@class='title style-scope ytmusic-two-row-item-renderer']")

album_links = [el.get_attribute("href") for el in album_elements]
album_names = [el.text for el in album_titles]

# Lặp qua từng album
for idx, album_link in enumerate(album_links):
    album_name = album_names[idx] if idx < len(album_names) else "Unknown Album"
    driver.get(album_link)
    time.sleep(3)
    
    try:
        tracks = driver.find_elements(By.XPATH, "//yt-formatted-string[@class='title style-scope ytmusic-responsive-list-item-renderer complex-string']//a")
        for track in tracks:
            track_title = track.text  # Thay 'title' thành 'tracklist'
            href = track.get_attribute("href")
            if href and 'watch?v=' in href:
                list_id = None
                if 'list=' in href:
                    list_id = href.split('list=')[1].split('&')[0]
                    album_map[list_id] = album_name  # Gán ánh xạ album cho list_id
                data.append({
                    "album": album_name,
                    "tracklist": track_title,  # Đổi từ 'title' thành 'tracklist'
                    "video_url": href,
                })
    except Exception as e:
        print(f"Lỗi khi lấy bài hát từ album {album_link}: {e}")

# Đóng trình duyệt
driver.quit()

# Tạo DataFrame
df_yt = pd.DataFrame(data)

# Lưu ra file Excel
df_yt.to_excel("youtube_music_tracks.xlsx", index=False)


Đã click 'Show all'


ZINGMP3+SPOTIFY+APPLE+YT MUSIC

In [19]:
# Chuẩn hóa tên cột để khớp
df_yt['tracklist'] = df_yt['tracklist'].apply(normalize_text)
df_yt['album'] = df_yt['album'].apply(normalize_text)

df_final['tracklist(danh sách bài hát)'] = df_final['tracklist(danh sách bài hát)'].apply(normalize_text)
df_final['album_name'] = df_final['album_name'].apply(normalize_text)
# Gộp hai bảng theo tên bài hát và tên album
df_merged = pd.merge(
    df_final, 
    df_yt, 
    left_on=["tracklist(danh sách bài hát)", "album_name"], 
    right_on=["tracklist", "album"], 
    how="left"
)
# Đổi tên cột video_url thành "YouTube Music"
df_merged = df_merged.rename(columns={"video_url": "YouTube Music"})

# Xóa các cột dư thừa
df_merged.drop(columns=["tracklist", "album"], inplace=True, errors='ignore')
df_merged


,album_name,Nghệ sĩ sở hữu album,album_type,genre,tracklist(danh sách bài hát),Ngày phát hành trên Spotify,Ngày phát hành trên ZingMP3,Ngày phát hành trên AppleMusic,Song artist(nghệ sĩ tham gia bài hát)(Spotify),Song artist(nghệ sĩ tham gia bài hát)(ZingMP3),Song artist(nghệ sĩ tham gia bài hát)(AppleMusic),Cung cấp bởi(ZingMP3),Cung cấp bởi(Spotify),Cung cấp bởi(AppleMusic),ZingMP3,Spotify,Apple Music,YouTube Music
0,Adamn,Bình Gold,Single,NaN,Adamn,10/04/2025,10/04/2025,NaN,None,Bình Gold,NaN,MIXUS,Binh Gold,NaN,https://zingmp3.vn/bai-hat/ADAMN-Binh-Gold/Z8U...,https://open.spotify.com/track/50WjeoNacrSiDum...,NaN,NaN
1,Bcdbl,Bình Gold,Single,NaN,Bcdbl,02/02/2022,NaN,NaN,None,NaN,NaN,NaN,Binh Gold,NaN,NaN,https://open.spotify.com/track/1Vi2Hln3U0kwCob...,NaN,https://music.youtube.com/watch?v=1POdjPZwZlg&...
2,Bcdbl,Bình Gold,Single,NaN,Bcdbl,02/02/2022,NaN,NaN,None,NaN,NaN,NaN,Binh Gold,NaN,NaN,https://open.spotify.com/track/1Vi2Hln3U0kwCob...,NaN,https://music.youtube.com/watch?v=1POdjPZwZlg&...
3,Bật Chế Độ Bay Lên (Huy Lee Remix),Bình Gold,Single,NaN,Bật Chế Độ Bay Lên (Short Version) - Huy Lee R...,04/07/2022,NaN,NaN,None,NaN,NaN,NaN,Bình Gold,NaN,NaN,https://open.spotify.com/track/7dPvKrYoFMd8QTs...,NaN,NaN
4,Bật Chế Độ Bay Lên (Huy Lee Remix),Bình Gold,Single,NaN,Bật Chế Độ Bay Lên - Huy Lee Remix,07/04/2022,NaN,NaN,None,NaN,NaN,NaN,Bình Gold,NaN,NaN,https://open.spotify.com/track/3eZcAYvAIVnRIJ8...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,Đổi Tư Thế (Rmx),Bình Gold,Single,NaN,Đổi Tư Thế - Rmx,17/01/2025,NaN,NaN,"Andree Right Hand, Trendy Nhân, Vũ Tarus",NaN,NaN,NaN,Trendy'N Records,NaN,NaN,https://open.spotify.com/track/5xczb6LcTWdehdF...,NaN,NaN
79,NaN,NaN,NaN,"Hip-Hop/Rap, Music",2K Trong Vali,NaN,NaN,31/12/2023,NaN,NaN,Dlow & Gxxfy,NaN,NaN,MaiDao Music,NaN,NaN,https://music.apple.com/us/album/2k-trong-vali...,NaN
80,Đổi Tư Thế (Single),NaN,Single,NaN,Đổi Tư Thế,NaN,02/10/2024,NaN,NaN,"Bình Gold, Andree Right Hand",NaN,MIXUS,NaN,NaN,https://zingmp3.vn/bai-hat/DOI-TU-THE-Binh-Gol...,NaN,NaN,NaN
81,Đớ,Bình Gold,Single,NaN,Đớ,04/03/2019,NaN,NaN,None,NaN,NaN,NaN,Binh Gold,NaN,NaN,https://open.spotify.com/track/6Dy4COn0KE2u4sZ...,NaN,NaN


In [20]:
df_merged.to_excel("song_ZingMP3+Spot+Apple+YTMusic.xlsx")